# Cross-Region Generalization Test: Tuktoyaktuk-Trained Model on Cambridge Bay

Runs `09`'s already-trained checkpoint (`s1_tuk_pcrtc_realattrs_spatialsplit_unet_best.pth`
-- the best leakage-free Tuktoyaktuk model) on a random subsample
(400 of 2101, seed 42) of Cambridge Bay's newly-extracted
Sentinel-1/LiDAR patches -- running all 2101 would cost ~10 hours of
DDIM sampling for a negligibly tighter estimate. **No retraining, no train/val
split** -- every Cambridge Bay patch is held-out test data. This answers
the actual question Michel asked: does what the model learned on
Tuktoyaktuk transfer to a region it has never seen?

**Confound to keep in mind when interpreting the result**: Cambridge
Bay's nearest available Sentinel-1 imagery is from May 2025, ~402-409
days after the LiDAR survey date and a different season (post-thaw
summer vs. the April/frozen-season survey) -- documented in `CONCEPTS.md`
and `11_cambridge_bay_patch_extraction.ipynb`. Any result here reflects
region generalization *and* this single season/date confound, not region
generalization alone. Still a single, clean confound -- not the
three-way one (region + date + polarization) that ruled out Pond Inlet.

## GPU configuration

In [ ]:
# Require CUDA -- this notebook only does inference, but still needs the GPU for the diffusion sampler
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import rasterio

assert torch.cuda.is_available(), 'CUDA is required. Run this notebook on the GPU environment.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0))

## Paths and configuration

In [ ]:
# Region-specific paths -- checkpoint comes from 09 (Tuktoyaktuk), data comes from Cambridge Bay
WORKING_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche')
TESSA_REPO = Path('/cs/student/project_msc/2025/aibh/jiayiche/tessa_baseline')
CHECKPOINT_NAME = 's1_tuk_pcrtc_realattrs_spatialsplit_unet_best.pth'
LIDAR_DIR = WORKING_REPO / 'input_data' / 'lidar_patches_cambridge_extracted' / 'lidar_patches_cambridge'
S1_DIR = WORKING_REPO / 'input_data' / 's1_patches_cambridge_pcrtc'
CHECKPOINT_DIR = WORKING_REPO / 'checkpoints'
OUTPUT_DIR = WORKING_REPO / 's1_training_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTEXT_K = 3
TARGET_HW = (256, 256)
BATCH_SIZE = 8
TIMESTEPS = 1000
NOISE_SCHEDULE = 'linear'
ATTENTION_VARIANT = 'default'
CAMBRIDGE_BAY_SURVEY_DATE = __import__('datetime').date(2024, 4, 18)

print('LIDAR_DIR:', LIDAR_DIR)
print('S1_DIR:', S1_DIR)
print('Checkpoint:', CHECKPOINT_DIR / CHECKPOINT_NAME)

## Import Tessa's baseline implementation

In [ ]:
# Import the model, scheduler, sampler, and metric functions from Tessa's baseline repo
sys.path.insert(0, str(TESSA_REPO))
from src.model.unet import ConditionalUNet
from src.diffusion.scheduler import LinearDiffusionScheduler, CosineDiffusionScheduler
from src.diffusion.sampling import p_sample_loop_ddim
from src.utils.recon_metrics import rmse, bias, sigma_error, normal_angle_error, average_jsd_multiscale, log_psd_rmse, zncc

## Dataset adapter -- identical to `09`'s real-attrs class, just no train/val split

Every patch is test data here, so there's no split logic at all --
`CambridgeBayS1Dataset` simply loads every matched patch.

In [ ]:
# Real-attrs dataset adapter, unchanged from 09 except the survey date used for age_norm
def build_real_attrs(s1_path, times, context_k, survey_date):
    attrs_path = s1_path / 'attrs.json'
    attrs_list = json.load(open(attrs_path)) if attrs_path.exists() else []
    vecs = []
    for time_path in times:
        idx = int(time_path.stem[1:])  # 't0' -> 0
        a = attrs_list[idx] if idx < len(attrs_list) else {}
        if a.get('acquisition_date'):
            import datetime as dt
            acq_date = dt.date.fromisoformat(a['acquisition_date'])
            age_days = (acq_date - survey_date).days
            age_norm = age_days / 30.0
        else:
            age_norm = 0.0
        orbit_dir = 1.0 if a.get('orbit_direction') == 'ASCENDING' else 0.0
        rel_orbit = (a.get('relative_orbit_number') or 0) / 175.0
        vecs.append([age_norm, orbit_dir, rel_orbit, 0.0, 0.0, 0.0, 0.0, 0.0])
    return torch.tensor(vecs, dtype=torch.float32).flatten()


class CambridgeBayS1Dataset(Dataset):
    def __init__(self, s1_dir, lidar_dir, patch_ids, context_k=3, target_hw=(256, 256), survey_date=None):
        self.s1_dir = Path(s1_dir)
        self.lidar_dir = Path(lidar_dir)
        self.patch_ids = list(patch_ids)
        self.context_k = context_k
        self.target_hw = target_hw
        self.survey_date = survey_date

    def __len__(self):
        return len(self.patch_ids)

    def __getitem__(self, index):
        patch_id = self.patch_ids[index]
        lidar_path = self.lidar_dir / f'lidar_patch_{patch_id}.tif'
        s1_path = self.s1_dir / f's1_patch_{patch_id}'
        with rasterio.open(lidar_path) as src:
            raw = src.read().astype(np.float32)
        target = raw[0]
        mask = (raw[1] > 0.5) if raw.shape[0] > 1 else np.isfinite(target)
        target = np.nan_to_num(target, nan=0.0, posinf=0.0, neginf=0.0)
        valid_count = max(1, int(mask.sum()))
        patch_mean = float(target[mask].sum() / valid_count)
        target = (target - patch_mean) * mask

        times = sorted(s1_path.glob('t*.tif'))[:self.context_k]
        if len(times) < self.context_k:
            raise RuntimeError(f'{s1_path} has fewer than {self.context_k} Sentinel-1 times')
        views = []
        for time_path in times:
            with rasterio.open(time_path) as src:
                sar = src.read()[:2].astype(np.float32)
            sar = np.nan_to_num(sar, nan=0.0, posinf=0.0, neginf=0.0)
            sar = np.maximum(sar, 1e-12)
            sar = 10.0 * np.log10(sar)
            sar_tensor = torch.from_numpy(sar).unsqueeze(0)
            sar_tensor = F.interpolate(sar_tensor, size=self.target_hw, mode='bilinear', align_corners=False).squeeze(0)
            sar_tensor = sar_tensor.repeat(2, 1, 1)
            views.append(sar_tensor)
        condition = torch.cat(views, dim=0)
        attrs = build_real_attrs(s1_path, times, self.context_k, self.survey_date)
        return {'lidar': torch.from_numpy(target).unsqueeze(0).float(), 'mask': torch.from_numpy(mask), 's1': condition.float(), 'attrs': attrs, 'patch_mean': torch.tensor(patch_mean), 'patch_id': patch_id}

## Build the test set -- a random subsample of matched Cambridge Bay patches, no split

2101 patches matched in `11`, but running DDIM inference on all of them
would take ~10 hours for a negligibly tighter mean-metric estimate than
a smaller sample -- standard error shrinks with 1/sqrt(N), so 400
patches (~7x Tuktoyaktuk's own 251-patch validation set) gives an
equally solid, citable number in a fraction of the time. No train/val
split either way -- every sampled patch is held-out test data.

In [ ]:
# All matched Cambridge Bay patches, then randomly subsampled to keep the DDIM inference run to a practical runtime
lidar_ids = {p.stem.split('_')[-1] for p in LIDAR_DIR.glob('lidar_patch_*.tif')}
s1_ids = {p.name.split('_')[-1] for p in S1_DIR.glob('s1_patch_*') if p.is_dir()}
all_test_ids = sorted(lidar_ids & s1_ids)
assert all_test_ids, 'No paired Cambridge Bay Sentinel-1/LiDAR patches found.'
print(f'Total matched Cambridge Bay patches available: {len(all_test_ids)}')

N_TEST_PATCHES = 400  # ~7x Tuktoyaktuk's own 251-patch validation set; running all 2101 would take ~10hrs of DDIM sampling for a negligibly tighter estimate
TEST_SAMPLE_SEED = 42
test_ids = random.Random(TEST_SAMPLE_SEED).sample(all_test_ids, min(N_TEST_PATCHES, len(all_test_ids)))
print(f'Subsampled to {len(test_ids)} patches for inference (seed={TEST_SAMPLE_SEED})')

test_dataset = CambridgeBayS1Dataset(S1_DIR, LIDAR_DIR, test_ids, CONTEXT_K, TARGET_HW, survey_date=CAMBRIDGE_BAY_SURVEY_DATE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## Load the Tuktoyaktuk-trained checkpoint (frozen, no further training)

In [ ]:
# Initialize the same architecture 09 used, then load its trained weights -- frozen, eval-only from here on
model = ConditionalUNet(in_channels=1, cond_channels=4 * CONTEXT_K, attr_dim=8 * CONTEXT_K, base_channels=128, embed_dim=256, unet_depth=4, attention_variant=ATTENTION_VARIANT, cond_k=CONTEXT_K).to(DEVICE)
scheduler = LinearDiffusionScheduler(TIMESTEPS, device=DEVICE) if NOISE_SCHEDULE == 'linear' else None

checkpoint = torch.load(CHECKPOINT_DIR / CHECKPOINT_NAME, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {checkpoint["epoch"]}, val_loss={checkpoint["val_loss"]:.6f}')
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## Run inference and compute reconstruction metrics

Identical metric suite to every other pcrtc evaluation in this project,
for direct comparison against Tuktoyaktuk's in-region numbers.

In [ ]:
# Run DDIM inference on every Cambridge Bay patch and compute the same reconstruction metrics used throughout this project
sampler = p_sample_loop_ddim
metric_rows = []
example_patches = []
N_EXAMPLES = 6
with torch.no_grad():
    for batch in test_loader:
        target = batch['lidar'].to(DEVICE)
        condition = batch['s1'].to(DEVICE)
        attrs = batch['attrs'].to(DEVICE)
        mask = batch['mask'].to(DEVICE).bool()
        prediction = sampler(model, scheduler, target.shape, condition, attrs, DEVICE)
        means = batch['patch_mean'].to(DEVICE).view(-1, 1, 1, 1)
        gt_absolute = target + means
        pred_absolute = prediction + means
        for i, patch_id in enumerate(batch['patch_id']):
            gt_i, pred_i, mask_i = gt_absolute[i], pred_absolute[i], mask[i]
            gt_valid = gt_i.squeeze()[mask_i].cpu().numpy()
            pred_valid = pred_i.squeeze()[mask_i].cpu().numpy()
            row = {
                'patch_id': patch_id,
                'rmse_m': float(rmse(gt_i, pred_i, mask_i).item()),
                'bias_m': float(bias(gt_i, pred_i, mask_i).item()),
                'sigma_error_pct': float(sigma_error(gt_i, pred_i, mask_i).item()),
                'normal_angle_error_deg': float(normal_angle_error(gt_i, pred_i, mask_i, pixel_size=1.0, degrees=True).item()),
                'jsd': float(average_jsd_multiscale(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'psd_rmse': float(log_psd_rmse(gt_i, pred_i, pixel_size=1.0, mask=mask_i).item()),
                'zncc': float(zncc(gt_i, pred_i, mask_i).item()),
                'gt_std_val': float(gt_valid.std()) if gt_valid.size > 0 else float('nan'),
                'pred_std_val': float(pred_valid.std()) if pred_valid.size > 0 else float('nan'),
            }
            metric_rows.append(row)
            if len(example_patches) < N_EXAMPLES:
                example_patches.append({
                    'patch_id': patch_id,
                    'gt': gt_i.squeeze().cpu().numpy(),
                    'pred': pred_i.squeeze().cpu().numpy(),
                    'mask': mask_i.squeeze().cpu().numpy(),
                    's1_condition': condition[i].cpu().numpy(),
                })

metrics_path = OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_metrics.json'
with metrics_path.open('w') as handle:
    json.dump(metric_rows, handle, indent=2)
print('Saved:', metrics_path)
print('Mean metrics:', {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'})

## GT-vs-pred residual standard deviation scatter

In [ ]:
# Per-patch GT-vs-pred residual std scatter, same diagnostic used in every other pcrtc notebook
import matplotlib.pyplot as plt
import pandas as pd

df_val = pd.DataFrame(metric_rows)
x = df_val['gt_std_val']
y = df_val['pred_std_val']

plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.5)
mn, mx = np.nanmin([x, y]), np.nanmax([x, y])
plt.plot([mn, mx], [mn, mx], 'r--', label='1:1')
mask_valid = ~np.isnan(x) & ~np.isnan(y)
z = np.polyfit(x[mask_valid], y[mask_valid], 1)
p = np.poly1d(z)
xs = np.array([mn, mx])
plt.plot(xs, p(xs), 'b-', label='Best fit')
r2 = np.corrcoef(x[mask_valid], y[mask_valid])[0, 1] ** 2
plt.text(0.05, 0.95, f'R\u00b2={r2:.3f}', transform=plt.gca().transAxes, va='top', bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel('Ground truth')
plt.ylabel('Prediction')
plt.title('Cambridge Bay Cross-Region LiDAR Residual Standard Deviation')
plt.legend()
plt.gca().set_aspect('equal', adjustable='box')
plt.savefig(OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_gt_pred_std_scatter.png', dpi=150, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_gt_pred_std_scatter.png')
plt.show()

## Reconstruction grid -- S1 views / GT / Pred / Error / PDF

In [ ]:
# Same reconstruction-grid style used throughout this project, for the 6 example Cambridge Bay patches
from matplotlib.colors import SymLogNorm

n_show = min(6, len(example_patches))
num_context_rows = CONTEXT_K
num_data_rows = 4
total_rows = num_context_rows + num_data_rows
num_cols = n_show

centered_gt = []
centered_pred = []
for p in example_patches[:n_show]:
    gt_mean = p['gt'][p['mask']].mean()
    pred_mean = p['pred'][p['mask']].mean()
    centered_gt.append(p['gt'] - gt_mean)
    centered_pred.append(p['pred'] - pred_mean)

all_gt = np.stack(centered_gt)
all_pred = np.stack(centered_pred)
max_abs_resid = np.quantile(np.abs(np.concatenate([all_gt.ravel(), all_pred.ravel()])), 0.995)
signed_error = all_pred - all_gt
max_abs_error = np.quantile(np.abs(signed_error.ravel()), 0.995)
norm = SymLogNorm(linthresh=0.1, linscale=1.0, vmin=-max_abs_resid, vmax=max_abs_resid, base=10)

row_titles = [f'S1 VV view {k + 1}' for k in range(CONTEXT_K)] + ['GT LiDAR (centered)', 'Pred LiDAR (centered)', 'Error', 'Patch PDF']

tile_size_inches = 4.0
fig, axes = plt.subplots(total_rows, num_cols, figsize=(num_cols * tile_size_inches + 2, total_rows * tile_size_inches), squeeze=False)

for col in range(num_cols):
    p = example_patches[col]
    gt_i, pred_i, mask_i = centered_gt[col], centered_pred[col], p['mask']
    s1_cond = p['s1_condition']

    axes[0, col].set_title(f"Patch {p['patch_id']}", fontsize=14, fontweight='bold')

    for k in range(CONTEXT_K):
        ax = axes[k, col]
        ax.imshow(s1_cond[k * 4], cmap='gray')
        ax.axis('off')

    row_gt = num_context_rows
    ax = axes[row_gt, col]
    ax.imshow(gt_i, cmap='RdBu_r', norm=norm)
    ax.axis('off')

    row_pred = num_context_rows + 1
    ax = axes[row_pred, col]
    ax.imshow(pred_i, cmap='RdBu_r', norm=norm)
    ax.axis('off')

    row_err = num_context_rows + 2
    err_i = pred_i - gt_i
    ax = axes[row_err, col]
    ax.imshow(err_i, cmap='seismic', vmin=-max_abs_error, vmax=max_abs_error)
    ax.axis('off')

    row_pdf = num_context_rows + 3
    ax = axes[row_pdf, col]
    gt_vals = gt_i[mask_i]
    pred_vals = pred_i[mask_i]
    if gt_vals.size > 1 and pred_vals.size > 1:
        all_vals = np.concatenate([gt_vals, pred_vals])
        vmin_pdf, vmax_pdf = np.quantile(all_vals, [0.01, 0.99])
        if np.isclose(vmin_pdf, vmax_pdf):
            vmin_pdf, vmax_pdf = all_vals.min(), all_vals.max()
        bins = np.linspace(vmin_pdf, vmax_pdf, 60)
        gt_pdf, edges = np.histogram(gt_vals, bins=bins, density=True)
        pred_pdf, _ = np.histogram(pred_vals, bins=bins, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, gt_pdf, color='darkblue', lw=2, label='GT')
        ax.plot(centers, pred_pdf, color='red', lw=2, alpha=0.95, label='Pred')
        ax.fill_between(centers, gt_pdf, color='darkblue', alpha=0.15)
        ax.fill_between(centers, pred_pdf, color='red', alpha=0.15)
        ax.set_xlim(vmin_pdf, vmax_pdf)
    ax.set_xlabel('Residual (m)', fontsize=10)
    if col == 0:
        ax.set_ylabel('Density', fontsize=10)
        ax.legend(fontsize=10, frameon=False)
    else:
        ax.set_yticklabels([])
    ax.set_box_aspect(1)

for row in range(total_rows):
    axes[row, 0].text(-0.25, 0.5, row_titles[row], ha='right', va='center', transform=axes[row, 0].transAxes, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_reconstructions_pdfs.png', dpi=200, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_reconstructions_pdfs.png')
plt.show()

## Metric violin plots

In [ ]:
# Distribution of each metric across all 2101 Cambridge Bay patches
import seaborn as sns

metrics_to_plot = [
    ('rmse_m', 'RMSE (m)', 'm'),
    ('bias_m', 'Bias (m)', 'm'),
    ('sigma_error_pct', 'RMS Height Error (%)', '%'),
    ('normal_angle_error_deg', 'Normal Angle Error (deg)', '\u00b0'),
    ('zncc', 'Cross Correlation (ZNCC)', ''),
    ('jsd', 'Distribution Divergence (JSD)', ''),
    ('psd_rmse', 'Log Spectral Error (RMSE)', ''),
]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4 * len(metrics_to_plot), 6))
colors = sns.color_palette('husl', len(metrics_to_plot))

for ax, (key, title, unit), color in zip(axes, metrics_to_plot, colors):
    values = df_val[key].dropna()
    sns.violinplot(y=values, ax=ax, color=color, inner='box')
    ax.set_title(title)
    ax.set_ylabel('')
    mean_val = values.mean()
    ax.set_xlabel(f'Mean: {mean_val:.2f}{unit}', fontsize=10, fontweight='bold')

plt.tight_layout()
out_path = OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_metric_violin_plots.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out_path)

## Qualitative success/failure case analysis

Same textured-block success/failure circling used throughout this
project -- an original design for this project, not a figure from
Tessa's dissertation (confirmed by checking her PDF directly).

In [ ]:
# Same success/failure block-circling analysis used throughout this project, applied to Cambridge Bay examples
def find_success_failure_regions(gt, pred, mask, block_size=64):
    """Split the patch into blocks; success = textured block with lowest error,
    failure = textured block with highest error. 'Textured' excludes flat/boring
    background blocks so the comparison is about real roughness, not empty ice."""
    error = np.abs(pred - gt)
    h, w = gt.shape
    block_stats = []
    for i in range(h // block_size):
        for j in range(w // block_size):
            r0, r1 = i * block_size, (i + 1) * block_size
            c0, c1 = j * block_size, (j + 1) * block_size
            block_mask = mask[r0:r1, c0:c1]
            if block_mask.sum() < 0.5 * block_mask.size:
                continue
            block_stats.append({
                'row': r0 + block_size // 2, 'col': c0 + block_size // 2,
                'gt_std': gt[r0:r1, c0:c1][block_mask].std(),
                'error': error[r0:r1, c0:c1][block_mask].mean(),
            })
    if not block_stats:
        return None, None
    median_std = np.median([b['gt_std'] for b in block_stats])
    textured = [b for b in block_stats if b['gt_std'] >= median_std] or block_stats
    success = min(textured, key=lambda b: b['error'])
    failure = max(textured, key=lambda b: b['error'])
    return success, failure


n_show = min(6, len(example_patches))
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4.5))
if n_show == 1:
    axes = [axes]

for ax, patch in zip(axes, example_patches[:n_show]):
    gt, pred, mask = patch['gt'], patch['pred'], patch['mask']
    gt_display = gt - gt[mask].mean()

    ax.imshow(gt_display, cmap='terrain')
    ax.set_title(f"Patch {patch['patch_id']}")
    ax.axis('off')

    success, failure = find_success_failure_regions(gt, pred, mask)
    if success is not None:
        ax.add_patch(plt.Circle((success['col'], success['row']), 20, edgecolor='lime', facecolor='none', linewidth=2.5))
        ax.annotate('Success', (success['col'], success['row']), color='lime', fontsize=8, fontweight='bold',
                    xytext=(success['col'], success['row'] - 35), ha='center')
    if failure is not None:
        ax.add_patch(plt.Circle((failure['col'], failure['row']), 20, edgecolor='red', facecolor='none', linewidth=2.5))
        ax.annotate('Failure', (failure['col'], failure['row']), color='red', fontsize=8, fontweight='bold',
                    xytext=(failure['col'], failure['row'] + 45), ha='center')

plt.tight_layout()
out_path = OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_success_failure.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out_path)

## Comparison protocol

Compare this Cambridge Bay cross-region result against:
1. **`09`'s in-region Tuktoyaktuk validation** (`s1_pcrtc_realattrs_spatialsplit_validation_metrics.json`)
   -- the same model, tested on data from the region it was trained on.
   The gap between the two numbers reflects region generalization
   *combined with* the ~402-409 day / cross-season confound documented
   above, not region generalization in isolation.
2. **`08`'s unseen-date Tuktoyaktuk test** (same region, new date) -- if
   Cambridge Bay's drop is similar in size to `08`'s (ZNCC 0.52 to 0.01),
   that suggests the date/season confound dominates over anything
   region-specific. If Cambridge Bay holds up much better than `08`,
   that's a more encouraging sign for genuine spatial transfer despite
   the imperfect date match.

In [ ]:
# Compare against 09's in-region Tuktoyaktuk numbers to see how much the cross-region+cross-season test degrades performance
tuk_inregion_path = OUTPUT_DIR / 's1_pcrtc_realattrs_spatialsplit_validation_metrics.json'
new_mean = {key: float(np.nanmean([row[key] for row in metric_rows])) for key in metric_rows[0] if key != 'patch_id'}

if tuk_inregion_path.exists():
    tuk_rows = json.load(open(tuk_inregion_path))
    tuk_mean = {k: float(np.nanmean([r[k] for r in tuk_rows])) for k in tuk_rows[0] if k != 'patch_id'}
    print(f'{"metric":<20}{"Tuktoyaktuk in-region (09)":>28}{"Cambridge Bay cross-region":>28}')
    for k in new_mean:
        if k in tuk_mean:
            print(f'{k:<20}{tuk_mean[k]:>28.4f}{new_mean[k]:>28.4f}')
else:
    print('09 in-region metrics file not found -- compare manually against the documented 09 numbers.')

## Confidence map: does the model's uncertainty stay calibrated cross-region?

Same ensemble-sampling approach as `07` (run the sampler `N_SAMPLES`
times per patch, treat sample-to-sample std as a confidence map), but
applied here to test something `07` couldn't: **does the Tuktoyaktuk
checkpoint's uncertainty estimate remain meaningful on a region it was
never trained on?** A model can degrade in two different ways on new
data -- wrong-but-confident (dangerous, uncertainty doesn't help) or
wrong-and-appropriately-uncertain (safer, uncertainty still means
something even when accuracy drops). This calibration correlation is
the direct test of which one this is.

In [ ]:
# Ensemble sampling function -- identical to 07's, replicates one patch's conditioning N times along the batch dim
import random
CALIBRATION_SEED = 43
EXAMPLE_SEED = 44
N_SAMPLES = 5
N_EXAMPLE_PATCHES = 6
N_CALIBRATION_PATCHES = 40

def sample_ensemble(model, scheduler, sampler_fn, item, n_samples, device):
    target = item['lidar'].unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)
    condition = item['s1'].unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)
    attrs = item['attrs'].unsqueeze(0).repeat(n_samples, 1).to(device)
    with torch.no_grad():
        samples = sampler_fn(model, scheduler, target.shape, condition, attrs, device)
    mean_val = item['patch_mean'].item()
    samples_absolute = samples + mean_val
    pred_mean = samples_absolute.mean(dim=0).squeeze(0)
    pred_std = samples_absolute.std(dim=0).squeeze(0)
    gt_absolute = item['lidar'].squeeze(0) + mean_val
    return pred_mean.cpu().numpy(), pred_std.cpu().numpy(), gt_absolute.numpy(), item['mask'].numpy()

### Calibration check

Runs the ensemble on `N_CALIBRATION_PATCHES` Cambridge Bay patches (a
random subset of the full 2101, not all of them, to keep runtime
reasonable), pools all valid pixels, and computes the Pearson
correlation between predicted std and actual absolute error.

In [ ]:
# Random subset of Cambridge Bay patches for the calibration check, pooled pixel-level correlation between predicted std and |error|
calib_ids = random.Random(CALIBRATION_SEED).sample(range(len(test_dataset)), min(N_CALIBRATION_PATCHES, len(test_dataset)))

all_std = []
all_abs_error = []
calib_rows = []
for idx in calib_ids:
    item = test_dataset[idx]
    pred_mean, pred_std, gt, mask = sample_ensemble(model, scheduler, sampler, item, N_SAMPLES, DEVICE)
    mask_bool = mask.astype(bool)
    abs_error = np.abs(pred_mean - gt)
    all_std.append(pred_std[mask_bool])
    all_abs_error.append(abs_error[mask_bool])
    calib_rows.append({
        'patch_id': item['patch_id'],
        'mean_std': float(pred_std[mask_bool].mean()) if mask_bool.any() else float('nan'),
        'mean_abs_error': float(abs_error[mask_bool].mean()) if mask_bool.any() else float('nan'),
    })

all_std = np.concatenate(all_std)
all_abs_error = np.concatenate(all_abs_error)
correlation = float(np.corrcoef(all_std, all_abs_error)[0, 1])
print(f'Patches used for calibration: {len(calib_ids)} (N_SAMPLES={N_SAMPLES} each)')
print(f'Cambridge Bay cross-region pixel-level correlation between predicted std and |error|: {correlation:.4f}')
print(f'(Compare against 07\'s in-region Tuktoyaktuk result: 0.4079)')

calibration_result = {
    'checkpoint': CHECKPOINT_NAME,
    'region': 'cambridge_bay_crossregion',
    'n_samples': N_SAMPLES,
    'n_calibration_patches': len(calib_ids),
    'pixel_correlation_std_vs_abs_error': correlation,
    'per_patch': calib_rows,
}
calib_path = OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_uncertainty_calibration.json'
with calib_path.open('w') as handle:
    json.dump(calibration_result, handle, indent=2)
print('Saved:', calib_path)

### Qualitative confidence map for example patches

Same GT / Pred Mean / Error / Uncertainty grid style as `07`.

In [ ]:
# Same GT/Pred/Error/Uncertainty grid as 07
example_ids = random.Random(EXAMPLE_SEED).sample(range(len(test_dataset)), min(N_EXAMPLE_PATCHES, len(test_dataset)))
examples = []
for idx in example_ids:
    item = test_dataset[idx]
    pred_mean, pred_std, gt, mask = sample_ensemble(model, scheduler, sampler, item, N_SAMPLES, DEVICE)
    examples.append({'patch_id': item['patch_id'], 'gt': gt, 'pred_mean': pred_mean, 'pred_std': pred_std, 'mask': mask})

n_cols = len(examples)
fig, axes = plt.subplots(4, n_cols, figsize=(n_cols * 4.0 + 2, 4 * 4.0), squeeze=False)
row_titles = ['GT LiDAR (centered)', 'Pred Mean (centered)', 'Error', 'Uncertainty (std)']

centered_gt = [e['gt'] - e['gt'][e['mask']].mean() for e in examples]
centered_pred = [e['pred_mean'] - e['pred_mean'][e['mask']].mean() for e in examples]

all_gt = np.stack(centered_gt)
all_pred = np.stack(centered_pred)
max_abs_resid = np.quantile(np.abs(np.concatenate([all_gt.ravel(), all_pred.ravel()])), 0.995)
all_std_vals = np.concatenate([e['pred_std'].ravel() for e in examples])
std_vmax = np.quantile(all_std_vals, 0.99)

from matplotlib.colors import SymLogNorm
norm = SymLogNorm(linthresh=0.1, linscale=1.0, vmin=-max_abs_resid, vmax=max_abs_resid, base=10)

for col, ex in enumerate(examples):
    gt_i, pred_i = centered_gt[col], centered_pred[col]
    axes[0, col].set_title(f"Patch {ex['patch_id']}", fontsize=14, fontweight='bold')
    axes[0, col].imshow(gt_i, cmap='RdBu_r', norm=norm); axes[0, col].axis('off')
    axes[1, col].imshow(pred_i, cmap='RdBu_r', norm=norm); axes[1, col].axis('off')
    err = pred_i - gt_i
    axes[2, col].imshow(err, cmap='seismic', vmin=-max_abs_resid, vmax=max_abs_resid); axes[2, col].axis('off')
    im_std = axes[3, col].imshow(ex['pred_std'], cmap='viridis', vmin=0, vmax=std_vmax); axes[3, col].axis('off')

for row in range(4):
    axes[row, 0].text(-0.25, 0.5, row_titles[row], ha='right', va='center', transform=axes[row, 0].transAxes, fontsize=12, fontweight='bold')

fig.colorbar(im_std, ax=axes[3, :].tolist(), orientation='horizontal', fraction=0.05, pad=0.08, label='Std (m)')
plt.savefig(OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_uncertainty_map.png', dpi=200, bbox_inches='tight')
print('Saved:', OUTPUT_DIR / 's1_pcrtc_cambridgebay_crossregion_uncertainty_map.png')
plt.show()